# 15.4 - Agent Evaluation

Status: VERIFIED

## What Are We Solving?
Agents are non-deterministic and stateful. A single 'correct output' metric is insufficient — you must evaluate tool selection, reasoning steps, final outcomes, and cost/safety.

## Mental Model
Agent evaluation is about the trajectory, not just the destination: Task -> Plan -> Action -> Observation -> Action -> ... -> Result

In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
import time
import json
print("All imports OK")

All imports OK


## Trajectory Tracking

In [2]:
@dataclass
class AgentTrajectory:
    task: str
    steps: list = field(default_factory=list)
    final_result: str = ""
    total_tokens: int = 0
    total_time_s: float = 0.0

    def add_step(self, tool: str, input_text: str, output_text: str):
        self.steps.append({
            "tool": tool,
            "input": input_text[:100],
            "output": output_text[:100],
        })

    def evaluate(self, expected_result: str) -> dict:
        return {
            "completed": bool(self.final_result),
            "steps_taken": len(self.steps),
            "exact_match": self.final_result.strip().lower() == expected_result.strip().lower(),
            "total_tokens": self.total_tokens,
            "time_seconds": self.total_time_s,
        }

# Create example trajectory
trajectory = AgentTrajectory(task="Summarize file report.txt")
trajectory.add_step("read_file", "report.txt", "Q3 revenue grew 15%...")
trajectory.add_step("summarize", "Q3 revenue grew 15%...", "Revenue increased 15% in Q3.")
trajectory.final_result = "Revenue increased 15% in Q3."
trajectory.total_tokens = 450
trajectory.total_time_s = 2.3

result = trajectory.evaluate("Revenue grew 15% in Q3.")
print("Trajectory Evaluation:")
for k, v in result.items():
    print(f"  {k}: {v}")

Trajectory Evaluation:
  completed: True
  steps_taken: 2
  exact_match: False
  total_tokens: 450
  time_seconds: 2.3


## Agent Benchmark Suite

In [3]:
def evaluate_agent_run(tasks: list, agent_fn) -> dict:
    results = []
    for task in tasks:
        start = time.time()
        traj = agent_fn(task["query"])
        elapsed = time.time() - start
        
        results.append({
            "task": task["query"],
            "completed": bool(traj.final_result),
            "steps": len(traj.steps),
            "correct": traj.final_result.strip().lower() == task["expected"].strip().lower(),
            "time": round(elapsed, 3),
        })
    
    total = len(results)
    return {
        "completion_rate": sum(r["completed"] for r in results) / total,
        "accuracy": sum(r["correct"] for r in results) / total,
        "avg_steps": round(sum(r["steps"] for r in results) / total, 1),
        "avg_time": round(sum(r["time"] for r in results) / total, 3),
        "results": results,
    }

# Mock agent
def mock_agent(query: str) -> AgentTrajectory:
    traj = AgentTrajectory(task=query)
    traj.add_step("search", query, f"Found info about: {query}")
    traj.add_step("summarize", "info", f"Summary of {query}")
    traj.final_result = f"Answer for: {query}"
    traj.total_tokens = 300
    return traj

tasks = [
    {"query": "What is Python?", "expected": "Python is a programming language."},
    {"query": "What is Docker?", "expected": "Docker is a containerization platform."},
    {"query": "What is RAG?", "expected": "RAG retrieves documents before generating."},
]

bench = evaluate_agent_run(tasks, mock_agent)
print("Agent Benchmark Results:")
print(f"  Completion Rate: {bench['completion_rate']:.1%}")
print(f"  Accuracy: {bench['accuracy']:.1%}")
print(f"  Avg Steps: {bench['avg_steps']}")
print(f"  Avg Time: {bench['avg_time']}s")

Agent Benchmark Results:
  Completion Rate: 100.0%
  Accuracy: 0.0%
  Avg Steps: 2.0
  Avg Time: 0.0s


## Cost and Efficiency Analysis

In [4]:
# Cost analysis
def analyze_agent_cost(benchmark: dict, cost_per_1k_tokens: float = 0.003) -> dict:
    total_tokens = sum(r.get("tokens", 300) for r in benchmark["results"])
    total_cost = total_tokens * cost_per_1k_tokens / 1000
    
    return {
        "total_tokens": total_tokens,
        "total_cost_usd": round(total_cost, 6),
        "cost_per_task": round(total_cost / len(benchmark["results"]), 6),
        "tokens_per_task": total_tokens // len(benchmark["results"]),
    }

cost = analyze_agent_cost(bench)
print("Cost Analysis:")
for k, v in cost.items():
    print(f"  {k}: {v}")

Cost Analysis:
  total_tokens: 900
  total_cost_usd: 0.0027
  cost_per_task: 0.0009
  tokens_per_task: 300


In [5]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Completion and accuracy
axes[0].bar(["Completion", "Accuracy"], [bench["completion_rate"], bench["accuracy"]], color=['steelblue', 'seagreen'])
axes[0].set_title('Agent Performance')
axes[0].set_ylim(0, 1.1)
axes[0].grid(True, alpha=0.3)

# Steps per task
task_names = [r["task"][:15] for r in bench["results"]]
steps = [r["steps"] for r in bench["results"]]
axes[1].barh(task_names, steps, color='coral')
axes[1].set_title('Steps per Task')
axes[1].grid(True, alpha=0.3)

# Cost breakdown
axes[2].pie([cost["total_cost_usd"], 0.01], labels=["API Cost", "Other"], autopct='%1.1f%%', colors=['gold', 'lightgray'])
axes[2].set_title('Cost Distribution')

plt.tight_layout()
plt.savefig('agent_eval.png', dpi=100, bbox_inches='tight')
plt.show()
print("Visualization saved")

Visualization saved


C:\Users\PC\AppData\Local\Temp\ipykernel_3348\654661730.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# Verification
assert bench["completion_rate"] > 0.5, "Completion rate too low"
assert bench["avg_steps"] > 0, "Must have steps"
print("VERIFICATION PASSED: Phase 15.4 complete")

VERIFICATION PASSED: Phase 15.4 complete
